In [1]:
RUN_ID         = "20260526_132433/batch_size_256"
MODEL_PATH     = "/ros2_ws/core/models/sweeps/" + RUN_ID + "/"
DADS_PARAM     = MODEL_PATH + "dads_params.pkl"
CONFIG         = None        # None -> installed defaults.yaml; else path to an override YAML
ROBOT          = "parol6"
SCENE          = None
TASK_NAME      = "skill_conditioned"
SEED           = 42
EPISODE_LENGTH = 200         # rollout horizon (env steps per skill)
NUM_SKILLS     = 16
# Match training: q_phi was trained on STOCHASTIC-policy samples, so eval the
# same way. deterministic=True (mode actions) pushes trajectories off q_phi's
# training distribution and tanks the metrics.
DETERMINISTIC  = False
# Lower if the reward computation OOMs at large prior_samples. Peak GPU memory
# for the reward scales with REWARD_CHUNK * prior_samples (independent of B, T).
REWARD_CHUNK   = 256

In [2]:
import jax
import jax.numpy as jnp

from pprint import pprint
import time
import pickle

from core_rl.robot import resolve_robot
from core_rl.env import make_env
from core_rl.tools.helper import (_load_config,
                                   print_params,
                                   _build_sac_policy,
                                   _build_skill_dynamics,
                                   sample_eval_skills)
from core_rl.dads.skill_dynamics import log_prob, compute_dads_reward

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


### Load config

In [3]:
## ---------------- Load config ---------------------------
# CONFIG=None loads the installed defaults.yaml (same path train.py uses).
cfg = _load_config(CONFIG)
env_cfg = cfg["env"]
dads_cfg = cfg["algorithms"]["dads"]

## ---------------- Resolve task kwargs (skill_size auto-sync) ----------------
# Mirror train.py: the env's skill_size MUST equal algorithms.dads.skill_size,
# else observation_size desyncs from the trained policy/normalizer. Copy the
# dict so we never mutate the shared cfg.
task_kwargs = dict(env_cfg["task_kwargs"][TASK_NAME])
algo_skill_size = int(dads_cfg["skill_size"])
env_skill_size = task_kwargs.get("skill_size")
if env_skill_size is not None and int(env_skill_size) != algo_skill_size:
    raise ValueError(
        f"skill_size mismatch: dads.skill_size={algo_skill_size} "
        f"vs task_kwargs.skill_size={env_skill_size}"
    )
task_kwargs["skill_size"] = algo_skill_size

print("task_kwargs:", task_kwargs)
print("dads_cfg:")
pprint(dads_cfg)

## ---------------- Make Robot and Env --------------------
robot = resolve_robot(ROBOT, SCENE)

env = make_env(
    robot=robot,
    task_name=TASK_NAME,
    backend=env_cfg.get("backend", "mjx"),
    control_dt=env_cfg["control_dt"],
    physics_dt=env_cfg["physics_dt"],
    max_episode_steps=max(env_cfg["max_episode_steps"], EPISODE_LENGTH + 1),
    scene=SCENE,
    task_kwargs=task_kwargs,
)

## ---------------- Load trained params -------------------
with open(DADS_PARAM, "rb") as f:
    dads_param = pickle.load(f)
assert isinstance(dads_param, tuple) and len(dads_param) == 3, (
    "expected params to be a 3-tuple (normalizer, policy, skill_dynamics)"
)
norm_params, policy_params, q_phi_params = dads_param
sac_params = (norm_params, policy_params)   # matches make_policy(params[:2]) in training

## ---------------- Rebuild networks from dads_cfg --------
make_policy = _build_sac_policy(env, dads_cfg)
q_phi_net, skill_size, skill_dyn_obs = _build_skill_dynamics(env, dads_cfg)

print(f"  obs_size={int(env.observation_size)}  action_size={int(env.action_size)}")
print(f"  skill_size={skill_size}  skill_dyn_obs={skill_dyn_obs}")
print(f"  fixed_std={dads_cfg.get('fixed_std', 1.0)}  prior_samples={dads_cfg['prior_samples']}")

## --------------- Sample skills for evaluation -----------
rng = jax.random.PRNGKey(SEED)
rng, rng_skills, rng_rollout, rng_metrics = jax.random.split(rng, 4)
skills = sample_eval_skills(skill_size, NUM_SKILLS, rng_skills)
print(f"First 4 skills:\n{skills[:4]}")

task_kwargs: {'base_task': 'ee_tracking', 'restricted_obs_indices': [12, 13, 14], 'init_noise': 0.05, 'skill_size': 3}
dads_cfg:
{'batch_size': 256,
 'discounting': 0.99,
 'entropy_coef': 0.1,
 'fixed_std': 0.02,
 'grad_updates_per_step': 16,
 'learning_rate': 0.0003,
 'max_replay_size': 100000,
 'min_replay_size': 20000,
 'network_factory_kwargs': {'hidden_layer_sizes': [512, 512]},
 'normalize_observations': True,
 'num_eval_envs': 64,
 'num_mixture_components': 4,
 'prior_samples': 500,
 'reward_scaling': 1.0,
 'skill_dyn_hidden_layer_sizes': [512, 512],
 'skill_dyn_lr': 0.0003,
 'skill_dyn_train_steps': 8,
 'skill_size': 3,
 'tau': 0.005}


/core/lib/python3.12/site-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


  obs_size=18  action_size=6
  skill_size=3  skill_dyn_obs=3
  fixed_std=0.02  prior_samples=500
First 4 skills:
[[ 0.45532846  0.5757351  -0.63661146]
 [-0.47473955 -0.7785413  -0.5947385 ]
 [-0.36462307 -0.78885937 -0.14032078]
 [-0.03922033 -0.31949592 -0.3061459 ]]


In [ ]:
## ---------------- Verify loaded params match rebuilt nets ----------------
# Eval must rebuild the EXACT architectures train.py used; otherwise the loaded
# weights are silently applied to the wrong shapes and metrics diverge from
# training. Rebuild q_phi from config and assert every leaf shape matches the
# pickle; cross-check the policy/normalizer against the env dims.

def _assert_same_shapes(name, built, loaded):
    sb = jax.tree_util.tree_map(lambda x: tuple(x.shape), built)
    sl = jax.tree_util.tree_map(lambda x: tuple(x.shape), loaded)
    if sb != sl:
        raise AssertionError(f"{name} shape mismatch:\n  rebuilt={sb}\n  loaded ={sl}")
    print(f"  [OK] {name}: all leaf shapes match")

# q_phi: a freshly-initialised net must have identical structure to the pickle.
q_phi_init = q_phi_net.init(jax.random.PRNGKey(0))
_assert_same_shapes("q_phi", q_phi_init, q_phi_params)

# policy + normalizer: check against env observation/action sizes.
obs_size = int(env.observation_size)
act_size = int(env.action_size)
pol = policy_params["params"]
hidden_keys = sorted(k for k in pol if k.startswith("hidden_"))
assert norm_params.mean.shape == (obs_size,), \
    f"normalizer mean {norm_params.mean.shape} != (obs_size={obs_size},)"
assert pol[hidden_keys[0]]["kernel"].shape[0] == obs_size, \
    f"policy input {pol[hidden_keys[0]]['kernel'].shape[0]} != obs_size {obs_size}"
assert pol[hidden_keys[-1]]["kernel"].shape[1] == 2 * act_size, \
    f"policy output {pol[hidden_keys[-1]]['kernel'].shape[1]} != 2*act_size {2 * act_size}"
print(f"  [OK] normalizer/policy: obs_size={obs_size}, action_size={act_size} (output=2*act for mean+log_std)")
print("All param and architecture checks passed - eval nets match the trained checkpoint.")

  [OK] q_phi: all leaf shapes match
  [OK] normalizer/policy: obs_size=18, action_size=6 (output=2*act for mean+log_std)
All param / architecture checks passed - eval nets match the trained checkpoint.


### Jitted Rollout function

For n parralel evalutions. action are done by the policy

In [ ]:
def _build_rollout_fn(env, policy_fn, skill_size: int, episode_length: int,
                      deterministic: bool = False, same_init: bool = False):
    """Roll out NUM_SKILLS envs in parallel, one fixed skill each.

    deterministic=False (default) samples tanh(mu + sigma*eps) - the same
    distribution q_phi was trained on. same_init=False gives each skill its own
    reset (closer to training's AutoReset), True shares one initial state.
    """

    def _rollout(params, skills, rng):
        n = skills.shape[0]

        rng, rng_reset = jax.random.split(rng)

        if same_init:
            state = env.reset(rng_reset)
            states = jax.tree_util.tree_map(lambda x: jnp.broadcast_to(x[None], (n, *x.shape)), state)
        else:
            reset_keys = jax.random.split(rng_reset, n)
            states = jax.vmap(env.reset)(reset_keys)

        # Force fixed skills: replace z in both obs and info.
        # obs layout from SkillConditionedTask.reset:  [base_obs, z]
        new_obs = states.obs.at[..., -skill_size:].set(skills)
        new_info = dict(states.info)
        new_info["z"] = skills
        states = states.replace(obs=new_obs, info=new_info)

        policy = policy_fn(params, deterministic=deterministic)

        def _step(carry, _):
            state, key = carry
            key, sub = jax.random.split(key)

            sub_keys = jax.random.split(sub, n)
            action, _ = jax.vmap(policy)(state.obs, sub_keys)
            next_state = jax.vmap(env.step)(state, action)

            # SkillConditionedTask.step writes BOTH s_restricted (pre-step) and
            # s_next_restricted (post-step) into the RETURNED state's info - read
            # both from next_state so delta_s is a clean one-step delta, exactly
            # like training's acting.actor_step.
            record = {
                "s_restricted":      next_state.info["s_restricted"],
                "s_next_restricted": next_state.info["s_next_restricted"],
                "z":                 next_state.info["z"],
                "obs":               state.obs,
                "action":            action,
                "qpos":              next_state.pipeline_state.q,
            }
            return (next_state, key), record

        (_, _), traj = jax.lax.scan(_step, (states, rng), length=episode_length)
        traj = jax.tree_util.tree_map(lambda x: jnp.swapaxes(x, 0, 1), traj)  # (T,N,..)->(N,T,..)

        return traj

    return jax.jit(_rollout)

In [ ]:
print("Compiling + running rollouts...")
t0 = time.time()
rollout_fn = _build_rollout_fn(
    env, make_policy, skill_size,
    episode_length=EPISODE_LENGTH,
    deterministic=DETERMINISTIC,
    same_init=False,
)
traj = rollout_fn(sac_params, skills, rng_rollout)

jax.tree_util.tree_map(lambda x: x.block_until_ready(), traj)
print(f"Rollout done in {time.time() - t0:.1f}s "
      f"({NUM_SKILLS} skills x {EPISODE_LENGTH} steps, deterministic={DETERMINISTIC})")
print_params(traj)

## Metrics

Important, vmap over compute_dads_reward doesnt work because there is not enough gpu space on my gpu. It needs to be done sequential by flatting the array over the Batches

In [ ]:
def compute_metrics(
    traj: dict,
    q_phi_net,
    q_phi_params,
    skill_size: int,
    prior_samples: int,
    rng: jax.Array,
    chunk_size: int = 256,
) -> dict:
    """DADS eval metrics from a (B, T, ...) rollout.

    The DADS reward vmaps q_phi over `prior_samples` (L) alternative skills,
    which materialises an intermediate of shape (rows, L, hidden). vmapping
    over all B*T rows at once OOMs for large L, so we flatten the transitions
    and stream them through `compute_dads_reward` in slices of `chunk_size`.
    Peak memory ~ chunk_size * L, independent of B, T and large L - lower
    `chunk_size` if you still OOM.
    """
    s_restricted = traj["s_restricted"]            # (B, T, D)
    s_next_restricted = traj["s_next_restricted"]  # (B, T, D)
    z = traj["z"]                                  # (B, T, D_z)
    B, T, D = s_restricted.shape
    delta_s = s_next_restricted - s_restricted

    ## -- q_phi log-prob of observed deltas (no L dim -> cheap, vmap is fine) --
    def f(s, zz, d):
        return log_prob(q_phi_net, q_phi_params, s, zz, d)

    logp = jax.vmap(f)(s_restricted, z, delta_s)   # (B, T)
    mean_logp = logp.mean(-1)                      # (B,)

    ## -- DADS reward, chunked over flattened transitions ---------------------
    s_flat = s_restricted.reshape(-1, D)
    s_next_flat = s_next_restricted.reshape(-1, D)
    z_flat = z.reshape(-1, z.shape[-1])
    M = s_flat.shape[0]                            # = B * T

    r_chunks = []
    for start in range(0, M, chunk_size):
        end = min(start + chunk_size, M)
        # Per-chunk key: deterministic given `rng`, decorrelated across chunks.
        chunk_rng = jax.random.fold_in(rng, start)
        z_alts = jax.random.uniform(
            chunk_rng, (end - start, prior_samples, skill_size),
            minval=-1.0, maxval=1.0,
        )
        r_chunks.append(compute_dads_reward(
            q_phi_net, q_phi_params,
            s_flat[start:end], z_flat[start:end], s_next_flat[start:end],
            z_alts,
        ))
    r_dads = jnp.concatenate(r_chunks, axis=0).reshape(B, T)   # (B, T)

    ## -- skill dispersion per time step --------------------------------------
    # std across skills (axis 0), averaged over the restricted-state dims.
    disp_per_t = jnp.std(s_restricted, axis=0).mean(-1)        # (T,)
    mean_dispersion = disp_per_t.mean()
    end_dispersion = disp_per_t[-1]

    # r_dads ceiling is log(L+1); the normalised value is in [0, 1].
    r_ceiling = float(jnp.log(prior_samples + 1))
    mean_r = float(r_dads.mean())

    return {
        "scalars": {
            "mean_dispersion": float(mean_dispersion),
            "end_dispersion": float(end_dispersion),
            "mean_logp": float(logp.mean()),
            "mean_r_dads": mean_r,
            "r_dads_ceiling": r_ceiling,            # = log(L+1)
            "r_dads_normalized": mean_r / r_ceiling,  # 1.0 = perfectly discriminable
        },
        "per_timestep_dispersion": disp_per_t,        # (T,)
        "per_skill": {
            "r_dads": r_dads,                         # (B, T)
            "logp": logp,                             # (B, T)
            "mean_logp": mean_logp,                   # (B,)
            "mean_r_dads": r_dads.mean(-1),           # (B,)
        },
    }

In [ ]:
# prior_samples (L) is pulled from the SAME config used in training so the
# reward ceiling log(L+1) matches. chunk_size only affects memory, not results.
metrics = compute_metrics(
    traj,
    q_phi_net,
    q_phi_params,
    skill_size=skill_size,
    prior_samples=dads_cfg["prior_samples"],
    rng=rng_metrics,
    chunk_size=REWARD_CHUNK,
)

print_params(metrics)
pprint(metrics["scalars"])

In [ ]:
import matplotlib.pyplot as plt

r = metrics["per_skill"]["r_dads"]              # (B, T)
ceiling = metrics["scalars"]["r_dads_ceiling"]  # log(L+1)

print(f"skill 0 mean r_dads (t>=25): {float(r[0, 25:].mean()):.4f}")
print(f"all-skill mean r_dads:       {metrics['scalars']['mean_r_dads']:.4f} "
      f"(ceiling log(L+1)={ceiling:.4f}, "
      f"normalized={metrics['scalars']['r_dads_normalized']:.2%})")

plt.figure(figsize=(8, 3.5))
plt.plot(r.mean(0), label="mean over skills", linewidth=2)
plt.plot(r[0], alpha=0.4, label="skill 0")
plt.axhline(ceiling, color="red", linestyle="--", linewidth=1, label="ceiling log(L+1)")
plt.xlabel("step"); plt.ylabel("r_dads"); plt.legend(); plt.grid(alpha=0.3)
plt.show()